
**Data Access**

In [0]:
bronze_loaction = 'abfss://bronze@adeprojectstorageaccount.dfs.core.windows.net/'
silver_location = 'abfss://silver@adeprojectstorageaccount.dfs.core.windows.net/'


**Trip Type Data**

In [0]:
df_trip_type = spark.read.format('csv')\
    .option('header',True)\
    .option('inferschema',True)\
    .load(f'{bronze_loaction}Trip_Type')

In [0]:
df_trip_type.display()


**Trip Zone Data**

In [0]:
df_trip_zone = spark.read.format('csv').option('header',True).option('inferschema',True).load(f'{bronze_loaction}Trip_Zone')

In [0]:
df_trip_zone.display()


**Trip Data**

In [0]:
df_trip = spark.read.format('parquet')\
    .load(f'{bronze_loaction}trips_2025/trip-data')

In [0]:
df_trip.display()


##Data Transformation


**Trip Type Data**

In [0]:
df_trip_type_trans = df_trip_type

In [0]:
df_trip_type_trans.display()

In [0]:
df_trip_type_trans.withColumnRenamed('trip_type','trip_id').withColumnRenamed('description','trip_type').display()

In [0]:
df_trip_type_trans = df_trip_type_trans.withColumnRenamed('trip_type','trip_id').withColumnRenamed('description','trip_type')

In [0]:
df_trip_type_trans.display()

In [0]:
df_trip_type_trans.printSchema()

In [0]:
df_trip_type_trans.write.format('parquet')\
    .mode('append')\
        .option('path',f'{silver_location}Trip_Type')\
        .save()


**Trip Zone Data**

In [0]:
df_trip_zone_trans = df_trip_zone

In [0]:
df_trip_zone_trans.display()


In [0]:
df_trip_zone_trans.printSchema()

In [0]:
df_trip_zone_trans.select('Borough').distinct().display()

In [0]:
df_trip_zone_trans=df_trip_zone_trans.withColumnsRenamed({'LocationID':'trip_location_id','Borough':'tirp_borough','Zone':'trip_zone','service_zone':'trip_service_zone'})

In [0]:
from pyspark.sql.functions import *

df_trip_zone_trans=df_trip_zone_trans.withColumns({'trip_zone1':get(split('trip_zone', '/'),0),'trip_zone2':get(split('trip_zone', '/'),1)})

df_trip_zone_trans=df_trip_zone_trans.select('trip_location_id','tirp_borough','trip_zone1','trip_zone2','trip_service_zone')

In [0]:
df_trip_zone_trans.display()

In [0]:
df_trip_zone_trans.write.format('parquet')\
    .mode('append')\
        .option('path',f'{silver_location}Trip_Zone')\
        .save()



**Trip Data**

In [0]:
df_trip_trans = df_trip

In [0]:
df_trip_trans.display()

In [0]:
df_trip_trans.printSchema()

In [0]:
df_trip_trans.select('ehail_fee').distinct().display()

In [0]:
df_trip_trans= df_trip_trans.withColumn('store_and_fwd_flag',when(col('store_and_fwd_flag') == 'N', 'No')\
    .when(col('store_and_fwd_flag') == 'Y', 'Yes'))

In [0]:
df_trip_trans.printSchema()

In [0]:
df_trip_trans=df_trip_trans.withColumnsRenamed({'VendorID':'customer_id','lpep_pickup_datetime':'pickup_datetime','lpep_dropoff_datetime':'dropoff_datetime','store_and_fwd_flag':'store_and_fwd_flag','RatecodeID':'trip_rating','PULocationID':'pickup_location_id','DOLocationID':'dropoff_location_id','passenger_count':'passenger_count','trip_distance':'trip_distance','fare_amount':'trip_fare_amount','extra':'trip_extra_charge','mta_tax':'trip_mta_tax','tip_amount':'trip_tip_amount','tolls_amount':'trip_tolls_amount','improvement_surcharge':'trip_improvement_surcharge','total_amount':'trip_total_amount','payment_type':'payment_type','trip_type':'trip_type','congestion_surcharge':'trip_congestion_surcharge','cbd_congestion_fee':'trip_cbd_congestion_fee'}).drop('ehail_fee')


In [0]:

df_trip_trans.display()

In [0]:
df_trip_trans.write.format('parquet')\
    .mode('append')\
        .option('path',f'{silver_location}Trip_2025')\
        .save()